[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/pandera-certified/notebooks/day-03-schema-model-class-api.ipynb#scrollTo=f1a2b3c4)

---
# Day 3 · SchemaModel — Class-Based Schemas with `pa.DataFrameModel`
**certified-journeys / pandera-certified** · Day 3 · Class API & Config

> **Goal for today:** Define schemas as Python classes using `pa.DataFrameModel`, attach checks with `pa.Field()` and `@pa.check` classmethods, convert to `DataFrameSchema`, and control strictness with `class Config`.


In [ ]:
%pip install -q pandera


## Step 1 · SchemaModel vs DataFrameSchema — when to use each

`pa.DataFrameModel` (also exported as `pa.SchemaModel`) is the **class-based**
counterpart to `DataFrameSchema`. Both validate DataFrames; the difference is
in how you declare the schema.

| Feature | `DataFrameSchema` (dict API) | `DataFrameModel` (class API) |
|---|---|---|
| Declaration style | Dict of `pa.Column(...)` | Class with type-annotated attributes |
| Reuse / inheritance | Composition | Class inheritance |
| IDE support | Limited | Full mypy / pyright type-checking |
| Attached checks | Inline `Check(...)` | `pa.Field()` + `@pa.check` methods |
| Config options | Constructor kwargs | Inner `class Config` |

Official docs: https://pandera.readthedocs.io/en/stable/schema_models.html


In [ ]:
import pandera as pa
import pandas as pd
from pandera.typing import Series

# ---- Dict API (Day 1 style) ----
schema_dict = pa.DataFrameSchema({
    "amount":         pa.Column(float),
    "currency":       pa.Column(str),
    "transaction_id": pa.Column(int),
})

# ---- Class API (SchemaModel style) ----
class TransactionModel(pa.DataFrameModel):
    amount:         Series[float]   # type annotation → dtype
    currency:       Series[str]
    transaction_id: Series[int]

# Validate the same DataFrame with both
df = pd.DataFrame({
    "amount":         [100.0, 250.0, 50.0],
    "currency":       ["USD", "EUR", "GBP"],
    "transaction_id": [1001, 1002, 1003],
})

result_dict  = schema_dict.validate(df)
result_model = TransactionModel.validate(df)   # class method on the model

print("Dict API result shape:", result_dict.shape)
print("Model API result shape:", result_model.shape)
print("Both produce identical validated DataFrames.")


### What just happened?

- **`TransactionModel.validate(df)`** works exactly like `schema.validate(df)` — same output, same `SchemaError` on failure.
- Type annotations (`Series[float]`) map directly to Pandera dtype checks — no need to repeat `pa.Column(float)`.
- The class API enables IDE autocompletion on field names and supports static type checking with mypy.
- **When to prefer the class API:** shared schemas across modules, inheritance hierarchies, or any project already using dataclasses / Pydantic.


## Step 2 · `pa.Field()` — attaching checks to annotated fields

`pa.Field()` is the class-API equivalent of `pa.Column(dtype, Check(...))`. It
accepts the most common check constraints as keyword arguments so you don't need
to write full `Check(...)` objects for standard cases.

```python
pa.Field(ge=0)                     # value >= 0
pa.Field(le=100)                   # value <= 100
pa.Field(isin=['USD','EUR','GBP'])  # set membership
pa.Field(nullable=True)            # allow NaN
pa.Field(str_matches=r'^TXN-\d+$') # regex
```


In [ ]:
class TransactionModelV2(pa.DataFrameModel):
    amount:         Series[float] = pa.Field(ge=0)                          # amount >= 0
    currency:       Series[str]   = pa.Field(isin=["USD", "EUR", "GBP"])    # allowed currencies
    transaction_id: Series[int]   = pa.Field(gt=0)                          # id must be positive

# Valid data
df_valid = pd.DataFrame({
    "amount":         [100.0, 0.0, 250.0],   # 0.0 is fine (ge=0)
    "currency":       ["USD", "EUR", "GBP"],
    "transaction_id": [1, 2, 3],
})

validated = TransactionModelV2.validate(df_valid)
print("Valid transactions:")
print(validated)

# Bad data: unsupported currency and negative amount
df_bad = pd.DataFrame({
    "amount":         [-5.0, 100.0],          # negative amount
    "currency":       ["USD", "JPY"],          # JPY not in allowed set
    "transaction_id": [4, 5],
})

try:
    TransactionModelV2.validate(df_bad, lazy=True)
except pa.errors.SchemaErrors as e:
    print("\nAll failures:")
    print(e.failure_cases[["column", "check", "failure_case"]])


### What just happened?

- **`pa.Field(ge=0)`** is syntactic sugar for `pa.Column(float, pa.Check.greater_than_or_equal_to(0))` — more concise and co-located with the type annotation.
- Field keyword shortcuts: `ge`, `gt`, `le`, `lt`, `ne`, `isin`, `notin`, `nullable`, `str_matches`, `str_contains`, `str_length`.
- **Lazy validation on a model** works the same as on `DataFrameSchema` — pass `lazy=True` to collect all failures.
- The `column` column in `failure_cases` uses the Python attribute name (e.g. `currency`), not a re-mapped name.


## Step 3 · `@pa.check` classmethod — column-level custom validators

For checks that can't be expressed with `pa.Field()` keyword arguments, decorate
a classmethod with `@pa.check(column_name)`. The method receives the column
Series and must return a boolean scalar or boolean Series.

```python
@pa.check("amount")
@classmethod
def mean_not_too_high(cls, series: Series[float]) -> bool:
    return series.mean() <= 5000
```


In [ ]:
class TransactionModelV3(pa.DataFrameModel):
    amount:         Series[float] = pa.Field(ge=0)
    currency:       Series[str]   = pa.Field(isin=["USD", "EUR", "GBP"])
    transaction_id: Series[int]   = pa.Field(gt=0)

    @pa.check("amount")
    @classmethod
    def mean_within_threshold(cls, series: Series[float]) -> bool:
        """Flag batches where the average transaction is suspiciously large."""
        threshold = 5000.0
        mean_val = series.mean()
        if mean_val > threshold:
            raise pa.errors.SchemaError(
                schema=None,
                data=series,
                message=f"amount mean {mean_val:.2f} exceeds threshold {threshold}",
            )
        return True  # returning True also works (no exception needed)

# Normal batch — mean well below threshold
df_normal = pd.DataFrame({
    "amount":         [100.0, 200.0, 300.0],
    "currency":       ["USD", "USD", "EUR"],
    "transaction_id": [1, 2, 3],
})

result = TransactionModelV3.validate(df_normal)
print("Normal batch mean:", result["amount"].mean())
print("Passed validation.")

# Suspicious batch — mean above threshold
df_suspicious = pd.DataFrame({
    "amount":         [9000.0, 8000.0, 7500.0],
    "currency":       ["USD", "USD", "EUR"],
    "transaction_id": [4, 5, 6],
})

try:
    TransactionModelV3.validate(df_suspicious)
except pa.errors.SchemaError as e:
    print("\nSuspicious batch caught:", e)


### What just happened?

- **`@pa.check("amount")`** decorates a classmethod that receives the entire `amount` Series — a vectorized check defined inside the model class.
- The method can either **return `bool`** (True = pass, False = fail) or **raise `SchemaError`** directly with a custom message.
- This is ideal for business-logic checks that depend on aggregate statistics (mean, sum, count) rather than per-row values.
- **The classmethod decorator order matters:** `@pa.check(...)` must come before `@classmethod`.


## Step 4 · `to_schema()` — converting a model to DataFrameSchema

Every `DataFrameModel` can be converted to a `DataFrameSchema` object using
`.to_schema()`. This is useful when you need to:

- Inspect the generated schema programmatically
- Combine a model with a dynamically-built schema
- Pass the schema to code that only accepts `DataFrameSchema`

```python
schema = MyModel.to_schema()   # → DataFrameSchema
schema.columns                 # dict of column names → Column objects
```


In [ ]:
# Convert TransactionModelV2 to DataFrameSchema and inspect it
schema = TransactionModelV2.to_schema()

print("Type:", type(schema))
print("\nColumns in generated schema:")
for name, col in schema.columns.items():
    print(f"  {name}: dtype={col.dtype}, nullable={col.nullable}")

# The schema works just like a hand-written DataFrameSchema
df = pd.DataFrame({
    "amount":         [10.0, 20.0],
    "currency":       ["USD", "EUR"],
    "transaction_id": [1, 2],
})

result = schema.validate(df)
print("\nValidated via to_schema():")
print(result)


### What just happened?

- **`to_schema()`** materialises the class definition into a concrete `DataFrameSchema` — the same object you'd write manually in the dict API.
- `schema.columns` is a dict of column names to `Column` objects — inspect dtype, nullable, and checks programmatically.
- This round-trip means **model and dict schemas are interchangeable** — you can start with a model and hand off a `DataFrameSchema` to legacy code.
- Useful for debugging: print `schema` to see the full textual representation of your model's constraints.


## Step 5 · `class Config` — coerce, strict, and other schema-wide settings

An inner `class Config` on your model sets schema-wide options, equivalent to
constructor kwargs on `DataFrameSchema`.

| Config option | Effect |
|---|---|
| `coerce = True` | Auto-cast all columns to declared dtypes |
| `strict = True` | Reject DataFrames with **extra columns** |
| `ordered = True` | Columns must appear in declaration order |
| `name = "MySchema"` | Label used in error messages |

**`strict=True`** is particularly useful at pipeline boundaries — it ensures no
unexpected columns sneak through.


In [ ]:
class StrictTransactionModel(pa.DataFrameModel):
    amount:         Series[float] = pa.Field(ge=0)
    currency:       Series[str]   = pa.Field(isin=["USD", "EUR", "GBP"])
    transaction_id: Series[int]   = pa.Field(gt=0)

    class Config:
        coerce = True    # auto-cast string inputs to correct dtypes
        strict = True    # reject any extra columns
        name   = "StrictTransactionModel"

# Data with string-typed columns (coerce=True will fix them)
df_coerce = pd.DataFrame({
    "amount":         ["100.0", "200.0"],   # strings
    "currency":       ["USD", "EUR"],
    "transaction_id": ["1", "2"],           # strings
})

result = StrictTransactionModel.validate(df_coerce)
print("After coerce=True, dtypes:")
print(result.dtypes)

# Now add an extra column — strict=True should reject it
df_extra = pd.DataFrame({
    "amount":         [100.0],
    "currency":       ["USD"],
    "transaction_id": [1],
    "extra_column":   ["should fail"],   # not declared in model
})

try:
    StrictTransactionModel.validate(df_extra)
except pa.errors.SchemaError as e:
    print("\nExtra column caught by strict=True:")
    print(e)


### What just happened?

- **`coerce=True` in Config** applies to all columns — the validated DataFrame has proper `float64` and `int64` dtypes even though the input contained strings.
- **`strict=True`** causes an immediate `SchemaError` if any column in the DataFrame is not declared in the model — extra columns are rejected, not silently ignored.
- Use `strict=True` at **sink boundaries** (writing to a database table, passing to a model) where an unexpected column could indicate a schema drift upstream.
- `strict = 'filter'` is a third option that silently **drops** extra columns instead of raising — useful for projection (keeping only declared columns).


In [ ]:
# Challenge: Define a ShipmentModel with these requirements:
#
#   Fields:
#     - shipment_id:   int, must be > 0
#     - origin:        str, must be one of ['US', 'EU', 'APAC']
#     - weight_kg:     float, must be >= 0.1 and <= 10000
#     - tracking_code: str, must match r'^SHP-[A-Z]{2}-\d{4}$'
#
#   Custom @pa.check on weight_kg:
#     - The batch total weight (sum) must not exceed 50000 kg
#
#   Config:
#     - coerce = True
#     - strict = True
#
# Then validate these two DataFrames:
#   1. A valid batch — should pass
#   2. A batch with total weight > 50000 — should fail the custom check

# class ShipmentModel(pa.DataFrameModel):
#     ...
#     class Config:
#         ...

# df_valid_shipments = pd.DataFrame({...})
# df_heavy_batch = pd.DataFrame({...})

# Test both and print results


---
## Day 3 key concepts recap

| Concept | What to remember |
|---|---|
| `pa.DataFrameModel` | Class-based schema; inherits for reuse; same validation as dict API |
| `Series[dtype]` annotation | Maps directly to column dtype declaration |
| `pa.Field(ge=0, isin=[...])` | Inline check constraints on annotated fields |
| `@pa.check("col")` | Classmethod validator; receives Series; return bool or raise SchemaError |
| `.to_schema()` | Convert model to `DataFrameSchema` for programmatic inspection |
| `class Config: coerce = True` | Schema-wide dtype casting |
| `class Config: strict = True` | Reject DataFrames with extra columns |

> **Tip:** Prefer `DataFrameModel` in new code — it's the direction Pandera is evolving, with better static typing support. Use `DataFrameSchema` when you need dynamic column names computed at runtime.

---
## What's next
**Day 4** → Index validation, multi-index schemas, and validating DataFrame-returning functions with `@pa.check_input` / `@pa.check_output` decorators.

Mark Day 3 complete in your [tracker](../index.html).
